# Autonomous Delivery Robot Search and Planning System

This notebook implements the delivery-robot problem from the Project 1 proposal using:
- Breadth-First Search (BFS)
- Uniform-Cost Search (UCS)
- A* Search

It also records solution length, path cost, generated nodes, expanded nodes, maximum frontier size, runtime, and returned action sequence.

## Problem Definition

State: `(robot_location, package_status, battery_level)`

Initial state: `('Depot', 'at_depot', 5)`

Goal: robot is at `Customer` and package status is `delivered`.

In [1]:
from collections import deque
import heapq
import itertools
import time

# The delivery map stores every location and the places directly connected to it.
# Each connection uses the format:
# "Destination": (travel_cost, battery_used)
MAP = {
    "Depot": {
        "Main Street": (2, 2),
        "Park": (3, 3)
    },
    "Main Street": {
        "Depot": (2, 2),
        "Charging Station": (2, 2),
        "Customer": (4, 4)
    },
    "Park": {
        "Depot": (3, 3),
        "Charging Station": (2, 2)
    },
    "Charging Station": {
        "Main Street": (2, 2),
        "Park": (2, 2),
        "Customer": (2, 2)
    },
    "Customer": {
        "Main Street": (4, 4),
        "Charging Station": (2, 2)
    }
}

# The robot can hold at most 6 battery units.
FULL_BATTERY = 6

# State format:
# (robot_location, package_status, battery_level)
INITIAL_STATE = ("Depot", "at_depot", 5)


In [2]:
class DeliveryRobotProblem:
    def __init__(self, initial_state=INITIAL_STATE):
        # Store the starting state for the search problem.
        self.initial_state = initial_state

    def actions(self, state):
        # Break the state into its three parts.
        location, package_status, battery = state

        # This list will hold every legal action available from the current state.
        actions = []

        # The robot can pick up the package only at the Depot
        # while the package is still waiting there.
        if location == "Depot" and package_status == "at_depot":
            actions.append(("PickUp", None))

        # The robot can recharge only at the Charging Station.
        # Recharging is unnecessary when the battery is already full.
        if location == "Charging Station" and battery < FULL_BATTERY:
            actions.append(("Recharge", None))

        # The robot can deliver only when it is at the Customer
        # and is currently carrying the package.
        if location == "Customer" and package_status == "carried":
            actions.append(("Deliver", None))

        # Check every location that is directly connected to the robot's
        # current location.
        for destination, (_, battery_used) in MAP[location].items():

            # Moving is legal only if the robot has enough battery
            # to make the trip.
            if battery >= battery_used:
                actions.append(("Move", destination))

        return actions

    def result(self, state, action):
        # Reject an action if it is not legal in the current state.
        if action not in self.actions(state):
            raise ValueError(f"Illegal action {action} from state {state}")

        location, package_status, battery = state
        action_type, destination = action

        # Picking up the package changes only the package status.
        if action_type == "PickUp":
            return (location, "carried", battery)

        # Recharging restores the battery to full capacity.
        if action_type == "Recharge":
            return (location, package_status, FULL_BATTERY)

        # Delivering changes the package status to delivered.
        if action_type == "Deliver":
            return (location, "delivered", battery)

        # Moving changes the robot's location and reduces the battery.
        if action_type == "Move":
            _, battery_used = MAP[location][destination]
            return (destination, package_status, battery - battery_used)

        raise ValueError("Unknown action")

    def is_goal(self, state):
        # The problem is solved only when the robot is at the Customer
        # and the package has been delivered.
        location, package_status, _ = state
        return location == "Customer" and package_status == "delivered"

    def step_cost(self, state, action, next_state):
        # Return the cost of the action that was taken.
        action_type, destination = action

        if action_type == "Move":
            # Movement cost comes from the road travel cost in MAP.
            return MAP[state[0]][destination][0]

        if action_type == "PickUp":
            return 1

        if action_type == "Recharge":
            return 3

        if action_type == "Deliver":
            return 1

        raise ValueError("Unknown action")


## Heuristic for A*

The heuristic estimates remaining cost while ignoring battery-related detours.  
That keeps the estimate optimistic.

In [3]:
def shortest_travel_costs():
    # Create a list containing every location in the map.
    locations = list(MAP.keys())

    # Start every distance at infinity.
    distances = {
        a: {b: float("inf") for b in locations}
        for a in locations
    }

    # The distance from a location to itself is always 0.
    for a in locations:
        distances[a][a] = 0

        # Add the direct travel costs from the map.
        for b, (travel_cost, _) in MAP[a].items():
            distances[a][b] = travel_cost

    # Floyd-Warshall finds the cheapest travel cost between every pair
    # of locations in this small graph.
    for k in locations:
        for i in locations:
            for j in locations:
                distances[i][j] = min(
                    distances[i][j],
                    distances[i][k] + distances[k][j]
                )

    return distances


# Precompute the travel-only distances once so the heuristic can reuse them.
TRAVEL_DISTANCE = shortest_travel_costs()


def heuristic(state, problem):
    # The heuristic estimates the remaining cost to finish the delivery.
    # It intentionally ignores battery-related detours and recharge costs,
    # which keeps the estimate optimistic.
    location, package_status, battery = state

    # If the package is already delivered, there is no remaining cost.
    if package_status == "delivered":
        return 0

    # If the robot already has the package, estimate:
    # travel cost to Customer + Deliver action cost.
    if package_status == "carried":
        return TRAVEL_DISTANCE[location]["Customer"] + 1

    # If the package is still at the Depot, estimate:
    # cost to reach Depot
    # + PickUp cost
    # + travel from Depot to Customer
    # + Deliver cost
    return (
        TRAVEL_DISTANCE[location]["Depot"]
        + 1
        + TRAVEL_DISTANCE["Depot"]["Customer"]
        + 1
    )


In [4]:
def reconstruct(node):
    # Work backward from the goal node to recover the complete solution.
    actions = []
    states = []

    while node is not None:
        states.append(node["state"])

        # The root node has no action, so skip it.
        if node["action"] is not None:
            actions.append(node["action"])

        node = node["parent"]

    # The path was collected backward, so reverse it.
    return list(reversed(actions)), list(reversed(states))


def result_record(name, node, generated, expanded, max_frontier, runtime):
    # Store search statistics in one consistent dictionary.

    # If node is None, the search failed to find a solution.
    if node is None:
        return {
            "algorithm": name,
            "found": False,
            "actions": [],
            "states": [],
            "solution_length": 0,
            "path_cost": None,
            "generated_nodes": generated,
            "expanded_nodes": expanded,
            "max_frontier": max_frontier,
            "runtime_seconds": runtime
        }

    # Recover the action sequence and states for a successful search.
    actions, states = reconstruct(node)

    return {
        "algorithm": name,
        "found": True,
        "actions": actions,
        "states": states,
        "solution_length": len(actions),
        "path_cost": node["cost"],
        "generated_nodes": generated,
        "expanded_nodes": expanded,
        "max_frontier": max_frontier,
        "runtime_seconds": runtime
    }


In [5]:
def breadth_first_search(problem):
    # Record the starting time so runtime can be measured.
    start = time.perf_counter()

    # Create the root node for the initial state.
    root = {
        "state": problem.initial_state,
        "parent": None,
        "action": None,
        "cost": 0
    }

    # BFS uses a queue, so deque is a natural choice.
    frontier = deque([root])

    # Track states already in the frontier so duplicates are not added.
    frontier_states = {problem.initial_state}

    # Track states that have already been expanded.
    explored = set()

    # Statistics required by the assignment.
    generated = 1
    expanded = 0
    max_frontier = 1

    while frontier:
        # BFS removes the oldest node first.
        node = frontier.popleft()
        state = node["state"]

        frontier_states.discard(state)

        # Stop immediately if the current state satisfies the goal.
        if problem.is_goal(state):
            return result_record(
                "BFS",
                node,
                generated,
                expanded,
                max_frontier,
                time.perf_counter() - start
            )

        # Skip the state if it has already been expanded.
        if state in explored:
            continue

        explored.add(state)
        expanded += 1

        # Generate every legal successor state.
        for action in problem.actions(state):
            child_state = problem.result(state, action)

            # Avoid repeated states.
            if child_state in explored or child_state in frontier_states:
                continue

            child = {
                "state": child_state,
                "parent": node,
                "action": action,
                "cost": node["cost"]
                + problem.step_cost(state, action, child_state)
            }

            frontier.append(child)
            frontier_states.add(child_state)
            generated += 1

        # Record the largest queue size reached during the search.
        max_frontier = max(max_frontier, len(frontier))

    # If the queue becomes empty, no solution was found.
    return result_record(
        "BFS",
        None,
        generated,
        expanded,
        max_frontier,
        time.perf_counter() - start
    )


In [6]:
def uniform_cost_search(problem):
    # Uniform-Cost Search always expands the lowest-cost path first.
    start = time.perf_counter()

    # The counter prevents Python from trying to compare dictionaries
    # when two frontier entries have the same path cost.
    counter = itertools.count()

    root = {
        "state": problem.initial_state,
        "parent": None,
        "action": None,
        "cost": 0
    }

    # Heap entries use:
    # (path_cost, tie_breaker, node)
    frontier = [(0, next(counter), root)]

    # Store the best path cost found so far for each state.
    best_cost = {problem.initial_state: 0}

    generated = 1
    expanded = 0
    max_frontier = 1

    while frontier:
        # Remove the frontier node with the smallest path cost.
        cost, _, node = heapq.heappop(frontier)
        state = node["state"]

        # Ignore old frontier entries that are no longer the cheapest
        # known way to reach this state.
        if cost != best_cost.get(state):
            continue

        if problem.is_goal(state):
            return result_record(
                "UCS",
                node,
                generated,
                expanded,
                max_frontier,
                time.perf_counter() - start
            )

        expanded += 1

        # Generate all legal successor states.
        for action in problem.actions(state):
            child_state = problem.result(state, action)

            # UCS adds the step cost to the current path cost.
            new_cost = (
                cost
                + problem.step_cost(state, action, child_state)
            )

            # Keep this path only if it improves the best known cost.
            if new_cost < best_cost.get(child_state, float("inf")):
                best_cost[child_state] = new_cost

                child = {
                    "state": child_state,
                    "parent": node,
                    "action": action,
                    "cost": new_cost
                }

                heapq.heappush(
                    frontier,
                    (new_cost, next(counter), child)
                )

                generated += 1

        max_frontier = max(max_frontier, len(frontier))

    return result_record(
        "UCS",
        None,
        generated,
        expanded,
        max_frontier,
        time.perf_counter() - start
    )


In [7]:
def a_star_search(problem, heuristic_function):
    # A* combines:
    # g(n) = path cost so far
    # h(n) = estimated remaining cost
    # f(n) = g(n) + h(n)
    start = time.perf_counter()

    counter = itertools.count()

    root = {
        "state": problem.initial_state,
        "parent": None,
        "action": None,
        "cost": 0
    }

    # The root priority is its heuristic value because g(n) starts at 0.
    root_priority = heuristic_function(
        problem.initial_state,
        problem
    )

    frontier = [
        (root_priority, next(counter), root)
    ]

    # Store the cheapest actual path cost found for each state.
    best_cost = {problem.initial_state: 0}

    generated = 1
    expanded = 0
    max_frontier = 1

    while frontier:
        # Expand the node with the smallest f(n) value.
        _, _, node = heapq.heappop(frontier)

        state = node["state"]
        cost = node["cost"]

        # Skip an outdated frontier entry if a cheaper path to this state
        # has already been found.
        if cost != best_cost.get(state):
            continue

        if problem.is_goal(state):
            return result_record(
                "A*",
                node,
                generated,
                expanded,
                max_frontier,
                time.perf_counter() - start
            )

        expanded += 1

        for action in problem.actions(state):
            child_state = problem.result(state, action)

            # g(n): true path cost to the child.
            new_cost = (
                cost
                + problem.step_cost(state, action, child_state)
            )

            # Keep only an improved path to the child state.
            if new_cost < best_cost.get(child_state, float("inf")):
                best_cost[child_state] = new_cost

                child = {
                    "state": child_state,
                    "parent": node,
                    "action": action,
                    "cost": new_cost
                }

                # h(n): estimated remaining cost.
                estimate = heuristic_function(
                    child_state,
                    problem
                )

                # f(n) = g(n) + h(n)
                priority = new_cost + estimate

                heapq.heappush(
                    frontier,
                    (priority, next(counter), child)
                )

                generated += 1

        max_frontier = max(max_frontier, len(frontier))

    return result_record(
        "A*",
        None,
        generated,
        expanded,
        max_frontier,
        time.perf_counter() - start
    )


## Required Tests

In [8]:
# Create the problem instance used for testing.
problem = DeliveryRobotProblem()

# Test the initial state.
print("Initial state:", problem.initial_state)

# Show the legal actions available from the starting state.
print("Initial legal actions:", problem.actions(problem.initial_state))

# Test a state where the robot has several movement choices.
multiple_action_state = ("Main Street", "carried", 6)

print("\nState with multiple legal actions:")
print(multiple_action_state)
print(problem.actions(multiple_action_state))

# Test the goal function with one true case and one false case.
print("\nGoal test true:")
print(problem.is_goal(("Customer", "delivered", 2)))

print("Goal test false:")
print(problem.is_goal(("Customer", "carried", 2)))

# Test an illegal action to make sure the program rejects it.
print("\nInvalid action test:")

try:
    problem.result(
        ("Depot", "at_depot", 5),
        ("Deliver", None)
    )
except ValueError as error:
    print(error)


Initial state: ('Depot', 'at_depot', 5)
Initial legal actions: [('PickUp', None), ('Move', 'Main Street'), ('Move', 'Park')]

State with multiple legal actions:
('Main Street', 'carried', 6)
[('Move', 'Depot'), ('Move', 'Charging Station'), ('Move', 'Customer')]

Goal test true:
True
Goal test false:
False

Invalid action test:
Illegal action ('Deliver', None) from state ('Depot', 'at_depot', 5)


In [9]:
# Test the heuristic on several different states.
print("Heuristic tests:")

test_states = [
    ("Depot", "at_depot", 5),
    ("Main Street", "carried", 3),
    ("Charging Station", "carried", 1),
    ("Customer", "delivered", 4)
]

for state in test_states:
    print(
        state,
        "->",
        heuristic(state, problem)
    )


Heuristic tests:
('Depot', 'at_depot', 5) -> 8
('Main Street', 'carried', 3) -> 5
('Charging Station', 'carried', 1) -> 3
('Customer', 'delivered', 4) -> 0


## Run and Compare BFS, UCS, and A*

In [10]:
# Run all three search algorithms on the exact same problem.
bfs_result = breadth_first_search(problem)
ucs_result = uniform_cost_search(problem)
astar_result = a_star_search(problem, heuristic)

results = [
    bfs_result,
    ucs_result,
    astar_result
]

def readable_action(action):
    # Make movement actions easier to read in the output.
    if action[0] == "Move":
        return f"Move({action[1]})"

    return action[0]

# Print the required performance measurements for each algorithm.
for r in results:
    print("=" * 60)
    print(r["algorithm"])
    print("Solution found:", r["found"])
    print("Solution length:", r["solution_length"])
    print("Path cost:", r["path_cost"])
    print("Generated nodes:", r["generated_nodes"])
    print("Expanded nodes:", r["expanded_nodes"])
    print("Maximum frontier:", r["max_frontier"])
    print(
        "Runtime:",
        f'{r["runtime_seconds"]:.8f}',
        "seconds"
    )

    print("Action sequence:")

    for i, action in enumerate(r["actions"], 1):
        print(
            f"  {i}. {readable_action(action)}"
        )


BFS
Solution found: True
Solution length: 6
Path cost: 11
Generated nodes: 32
Expanded nodes: 27
Maximum frontier: 9
Runtime: 0.00016367 seconds
Action sequence:
  1. PickUp
  2. Move(Main Street)
  3. Move(Charging Station)
  4. Recharge
  5. Move(Customer)
  6. Deliver
UCS
Solution found: True
Solution length: 6
Path cost: 11
Generated nodes: 30
Expanded nodes: 22
Maximum frontier: 9
Runtime: 0.00014946 seconds
Action sequence:
  1. PickUp
  2. Move(Main Street)
  3. Move(Charging Station)
  4. Recharge
  5. Move(Customer)
  6. Deliver
A*
Solution found: True
Solution length: 6
Path cost: 11
Generated nodes: 16
Expanded nodes: 8
Maximum frontier: 8
Runtime: 0.00007228 seconds
Action sequence:
  1. PickUp
  2. Move(Main Street)
  3. Move(Charging Station)
  4. Recharge
  5. Move(Customer)
  6. Deliver


In [11]:
# Build a compact comparison table using plain Python.

headers = [
    "Algorithm",
    "Found?",
    "Length",
    "Path Cost",
    "Generated",
    "Expanded",
    "Max Frontier",
    "Runtime (s)"
]

rows = []

# Convert each result dictionary into one table row.
for r in results:
    rows.append([
        r["algorithm"],
        r["found"],
        r["solution_length"],
        r["path_cost"],
        r["generated_nodes"],
        r["expanded_nodes"],
        r["max_frontier"],
        f'{r["runtime_seconds"]:.8f}'
    ])

# Find the width needed for each column.
widths = []

for i, header in enumerate(headers):
    widths.append(
        max(
            len(str(header)),
            *(len(str(row[i])) for row in rows)
        )
    )

def print_row(row):
    # Print one row with columns aligned.
    print(
        " | ".join(
            str(value).ljust(widths[i])
            for i, value in enumerate(row)
        )
    )

# Print the table header.
print_row(headers)

# Print a divider line.
print(
    "-+-".join(
        "-" * width
        for width in widths
    )
)

# Print the algorithm results.
for row in rows:
    print_row(row)


Algorithm | Found? | Length | Path Cost | Generated | Expanded | Max Frontier | Runtime (s)
----------+--------+--------+-----------+-----------+----------+--------------+------------
BFS       | True   | 6      | 11        | 32        | 27       | 9            | 0.00016367 
UCS       | True   | 6      | 11        | 30        | 22       | 9            | 0.00014946 
A*        | True   | 6      | 11        | 16        | 8        | 8            | 0.00007228 


## Planning Aspect

This is more than route finding because the robot must perform actions in a valid order.

- **PickUp** changes the package from `at_depot` to `carried`.
- **Move** changes location and lowers battery.
- **Recharge** restores the battery to 6.
- **Deliver** changes the package from `carried` to `delivered`.

Reaching the Customer without carrying and delivering the package does not satisfy the goal.